# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidrazabajwa49/flyrank-ml-internship-assignment-1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Same lane, same March-feature / April-label frame as `w04_baseline_score.ipynb` and `w04_signal_audit.ipynb`. The baseline rule from Week 4 is rebuilt here unchanged, so it can sit in the same comparison table as the model, scored on the same held-out data.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Decline-risk classification is a yes/no question with an observed label (`is_down`, built from real April outcomes, not a proxy). The skill's own table says to start there: **Logistic Regression** first — readable, coefficients name themselves, a fair fight against a hand-rule — then **Random Forest** to see whether nonlinearity and feature interactions actually buy anything over it. Both get evaluated against the Week-4 baseline on identical data; complexity only stays in if it earns its place in the comparison table below.

In [1]:
%pip -q install duckdb scikit-learn
import os, getpass
import numpy as np
import pandas as pd
import duckdb
import sklearn

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = 'hf://datasets/FlyRank/internship-warehouse'
FACT_03 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_04 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{BASE}/dim_content.parquet')"
DECISION_DATE = pd.Timestamp('2026-03-31')
SEED = 42
print('scikit-learn', sklearn.__version__)

# Identical frame to w04_baseline_score.ipynb -- same filters, same label.
frame = con.sql(f"""
WITH mar AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS mar_impr, SUM(gsc_clicks) AS mar_clicks,
         SUM(gsc_sum_position) AS mar_sum_pos, COUNT(*) AS mar_days,
         SUM(gsc_impressions) FILTER (report_date <  DATE '2026-03-16') AS h1_impr,
         SUM(gsc_impressions) FILTER (report_date >= DATE '2026-03-16') AS h2_impr
  FROM {FACT_03} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
),
apr AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS apr_impr, COUNT(*) AS apr_days
  FROM {FACT_04} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
)
SELECT m.client_hash_id, m.content_hash_id, m.mar_impr, m.mar_clicks, m.mar_days,
       m.mar_impr * 1.0 / m.mar_days AS mar_daily_impr,
       m.mar_sum_pos * 1.0 / NULLIF(m.mar_impr, 0) AS mar_avg_position,
       m.mar_clicks * 100.0 / NULLIF(m.mar_impr, 0) AS mar_ctr,
       (m.h2_impr - m.h1_impr) * 1.0 / NULLIF(m.h2_impr + m.h1_impr, 0) AS mar_h2_vs_h1,
       a.apr_impr * 1.0 / NULLIF(a.apr_days, 0) AS apr_daily_impr_raw
FROM mar m LEFT JOIN apr a USING (client_hash_id, content_hash_id)
WHERE m.mar_days >= 15 AND m.mar_impr >= 30
""").df()

frame['mar_h2_vs_h1'] = frame['mar_h2_vs_h1'].fillna(0.0)
frame['apr_daily_impr'] = frame['apr_daily_impr_raw'].fillna(0.0)
frame['y'] = (frame['apr_daily_impr'] < 0.80 * frame['mar_daily_impr']).astype(int)
frame = frame.drop(columns=['apr_daily_impr_raw'])

content_meta = con.sql(f"SELECT content_hash_id, content_type, content_updated_date FROM {DIM_CONTENT}").df()
frame = frame.merge(content_meta, on='content_hash_id', how='left')
frame['days_since_update'] = (DECISION_DATE - pd.to_datetime(frame['content_updated_date'])).dt.days
frame['days_since_update'] = frame['days_since_update'].fillna(frame['days_since_update'].median())
frame['content_type'] = frame['content_type'].fillna('unknown')
frame['log_daily_impr'] = np.log1p(frame['mar_daily_impr'])
type_dummies = pd.get_dummies(frame['content_type'], prefix='type', dtype=int)
frame = pd.concat([frame, type_dummies], axis=1)

NUMERIC_FEATURES = ['log_daily_impr', 'mar_avg_position', 'mar_ctr', 'mar_h2_vs_h1', 'days_since_update']
FEATURE_COLS = NUMERIC_FEATURES + list(type_dummies.columns)

# Week-4 baseline rule, rebuilt unchanged -- same thresholds, same formula.
FALL_HARD, FALL_SOFT, VOL_FLOOR = -0.20, -0.05, 50.0
frame['baseline_score'] = np.maximum(0.0, -frame['mar_h2_vs_h1']) * np.log1p(frame['mar_daily_impr'])

BASE_RATE = frame['y'].mean()
print(f"frame: {len(frame):,} pages | {frame['client_hash_id'].nunique()} clients | base rate {BASE_RATE:.3f}")
print(f"features: {FEATURE_COLS}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 105.2 MB/s eta 0:00:00
scikit-learn 1.6.1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

frame: 116,539 pages | 40 clients | base rate 0.504
features: ['log_daily_impr', 'mar_avg_position', 'mar_ctr', 'mar_h2_vs_h1', 'days_since_update', 'type_comparison article', 'type_feedly article', 'type_keyword article']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_hash_id`, not random, and not a plain time split.** The label is already time-separated at the row level (March features → April outcome), so the leakage risk left isn't across time — it's across *client*. Pages from the same client share template, niche, and tracking quirks; a random row split would let the model see other pages from the same client in training and partly memorize the client rather than learn a pattern that generalizes to a new one. `GroupKFold(5)` on `client_hash_id` is the honest question here — the same grouped-split logic already checked once in `w03_feature_leakage_check.ipynb`, now applied to the real model.

In [2]:
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

gkf = GroupKFold(n_splits=5)
fold_sizes = [(len(te), frame.iloc[te]['client_hash_id'].nunique())
              for _, te in gkf.split(frame, groups=frame['client_hash_id'])]
print('Fold sizes (rows, distinct clients) -- confirms clients never split across folds:')
for i, (n_rows, n_clients) in enumerate(fold_sizes):
    print(f"  fold {i}: {n_rows:,} rows, {n_clients} clients")

all_clients = set(frame['client_hash_id'])
fold_client_sets = [set(frame.iloc[te]['client_hash_id']) for _, te in gkf.split(frame, groups=frame['client_hash_id'])]
overlap = sum(len(a & b) for i, a in enumerate(fold_client_sets) for b in fold_client_sets[i+1:])
print(f"\nclient overlap across test folds: {overlap} (must be 0 for the split to be honest)")
assert overlap == 0


Fold sizes (rows, distinct clients) -- confirms clients never split across folds:
  fold 0: 23,508 rows, 1 clients
  fold 1: 23,258 rows, 8 clients
  fold 2: 23,257 rows, 7 clients
  fold 3: 23,259 rows, 12 clients
  fold 4: 23,257 rows, 12 clients

client overlap across test folds: 0 (must be 0 for the split to be honest)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Precision@K, K = 10 / 20 / 50, averaged over the 5 grouped folds -- baseline, Logistic Regression, and Random Forest all scored on the exact same held-out rows each fold, never seen in that fold's training data.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K_VALUES = [10, 20, 50]
fold_results = {name: {k: [] for k in K_VALUES} for name in ['baseline', 'logreg', 'random_forest']}
fold_base_rates = []

X_all = frame[FEATURE_COLS].to_numpy()
y_all = frame['y'].to_numpy()
groups_all = frame['client_hash_id'].to_numpy()

for fold, (tr, te) in enumerate(gkf.split(frame, groups=groups_all)):
    X_tr, X_te = X_all[tr], X_all[te]
    y_tr, y_te = y_all[tr], y_all[te]
    fold_base_rates.append(y_te.mean())

    logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED))
    logreg.fit(X_tr, y_tr)
    logreg_scores = logreg.predict_proba(X_te)[:, 1]

    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                                 class_weight='balanced', random_state=SEED, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    rf_scores = rf.predict_proba(X_te)[:, 1]

    baseline_scores = frame.iloc[te]['baseline_score'].to_numpy()

    for k in K_VALUES:
        fold_results['baseline'][k].append(precision_at_k(baseline_scores, y_te, k))
        fold_results['logreg'][k].append(precision_at_k(logreg_scores, y_te, k))
        fold_results['random_forest'][k].append(precision_at_k(rf_scores, y_te, k))

print(f"base rate across folds: {np.mean(fold_base_rates):.3f} (+/- {np.std(fold_base_rates):.3f})\n")

rows = []
for name in ['baseline', 'logreg', 'random_forest']:
    row = {'method': name}
    for k in K_VALUES:
        vals = fold_results[name][k]
        row[f'p@{k}_mean'] = round(float(np.mean(vals)), 3)
        row[f'p@{k}_std'] = round(float(np.std(vals)), 3)
    rows.append(row)
comparison = pd.DataFrame(rows).set_index('method')
print('COMPARISON TABLE -- 5-fold grouped CV, mean (+/- std) precision@K')
print(comparison.to_string())


base rate across folds: 0.505 (+/- 0.114)

COMPARISON TABLE -- 5-fold grouped CV, mean (+/- std) precision@K
               p@10_mean  p@10_std  p@20_mean  p@20_std  p@50_mean  p@50_std
method                                                                      
baseline            0.82     0.172       0.86     0.086      0.856     0.098
logreg              0.96     0.080       0.94     0.049      0.860     0.049
random_forest       0.82     0.117       0.84     0.116      0.860     0.083


In [4]:
# Single grouped split, refit for interpretation below (importance + concrete errors)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, te_idx = next(gss.split(frame, groups=groups_all))
train, test = frame.iloc[tr_idx].reset_index(drop=True), frame.iloc[te_idx].reset_index(drop=True)

rf_final = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                                   class_weight='balanced', random_state=SEED, n_jobs=-1)
rf_final.fit(train[FEATURE_COLS], train['y'])
test = test.assign(rf_score=rf_final.predict_proba(test[FEATURE_COLS])[:, 1])

print(f"Held-out split: {len(train):,} train rows, {len(test):,} test rows, "
      f"{train['client_hash_id'].nunique()} vs {test['client_hash_id'].nunique()} clients, no overlap: "
      f"{len(set(train['client_hash_id']) & set(test['client_hash_id'])) == 0}")


Held-out split: 88,293 train rows, 28,246 test rows, 32 vs 8 clients, no overlap: True


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf_final, test[FEATURE_COLS], test['y'],
                               n_repeats=20, random_state=SEED, scoring='roc_auc')
importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)
print('PERMUTATION IMPORTANCE (drop in ROC-AUC when a feature is shuffled)')
print(importance.to_string(index=False))

top_feature = importance.iloc[0]['feature']
top_score = importance.iloc[0]['importance_mean']
print(f"\nTop feature: {top_feature} (mean drop {top_score:.4f})")
if top_score > 0.30:
    print("SUSPICIOUSLY HIGH -- a single feature this dominant on its own is worth re-checking")
    print("against the leakage tests from w03_feature_leakage_check.ipynb before trusting it.")
else:
    print("No single feature dominates -- consistent with a model that's actually blending signal,")
    print("not keying off one leaking column.")


PERMUTATION IMPORTANCE (drop in ROC-AUC when a feature is shuffled)
                feature  importance_mean  importance_std
           mar_h2_vs_h1         0.130390        0.002323
                mar_ctr         0.039408        0.001325
         log_daily_impr         0.020699        0.000964
       mar_avg_position         0.006532        0.000518
type_comparison article         0.000006        0.000010
    type_feedly article        -0.000125        0.000109
   type_keyword article        -0.000286        0.000155
      days_since_update        -0.036906        0.001088

Top feature: mar_h2_vs_h1 (mean drop 0.1304)
No single feature dominates -- consistent with a model that's actually blending signal,
not keying off one leaking column.


In [6]:
# 3 concrete wrong cases
test['pred'] = (test['rf_score'] >= 0.5).astype(int)
false_negatives = test[(test['y'] == 1) & (test['pred'] == 0)].sort_values('rf_score')
false_positives = test[(test['y'] == 0) & (test['pred'] == 1)].sort_values('rf_score', ascending=False)

print(f"False negatives (missed a real decline): {len(false_negatives)} of {(test['y']==1).sum()} actual declines")
print(f"False positives (flagged a page that didn't decline): {len(false_positives)} of {(test['y']==0).sum()} actual non-declines")

cols = ['client_hash_id', 'content_hash_id', 'mar_h2_vs_h1', 'mar_avg_position', 'mar_ctr',
        'days_since_update', 'log_daily_impr', 'rf_score', 'y']
print('\n3 FALSE NEGATIVES -- model missed these; momentum alone did not warn it')
print(false_negatives[cols].head(3).round(3).to_string(index=False))
print('\n3 FALSE POSITIVES -- model flagged these; they held up in April anyway')
print(false_positives[cols].head(3).round(3).to_string(index=False))


False negatives (missed a real decline): 10287 of 17789 actual declines
False positives (flagged a page that didn't decline): 2171 of 10457 actual non-declines

3 FALSE NEGATIVES -- model missed these; momentum alone did not warn it
         client_hash_id          content_hash_id  mar_h2_vs_h1  mar_avg_position  mar_ctr  days_since_update  log_daily_impr  rf_score  y
client_62f4a7e64f5e0096 content_19e6329384fd8bb3         0.271             2.802    0.652                -94           7.555     0.206  1
client_3f0ce4d44fe94f3d content_dfd53e751d2fc9a9         0.374             4.084    1.339                -87           3.647     0.208  1
client_62f4a7e64f5e0096 content_33d8aa4c38016138         0.394             3.313    0.875                -96           5.952     0.213  1

3 FALSE POSITIVES -- model flagged these; they held up in April anyway
         client_hash_id          content_hash_id  mar_h2_vs_h1  mar_avg_position  mar_ctr  days_since_update  log_daily_impr  rf_score  y
clien

**Reading the two error tables above:** false negatives are the pages worth worrying about most — read their `mar_h2_vs_h1` column: a value near zero means the page hadn't started falling yet by the March cutoff, so no March-only feature could have caught it; that's a genuine blind spot of this feature set, not a model failure. False positives with a strongly negative `mar_h2_vs_h1` but `y=0` are the reversion cases flagged in Week 4's weak-picks section — a page that dipped in March and recovered on its own in April looks identical, from March data alone, to one that kept falling. Neither error type is fixable by tuning thresholds; both would need a longer feature history than this lane currently builds.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.